# ECERC with WavLM audio features (IEMOCAP)
Settings: Accelerator **GPU T4 x2**, Internet **On** (needed once to download WavLM from Hugging Face).
Inputs: `iemocap-ecerc-data`, `ecerc-code` (latest version), `iemocap-wav`.

In [ ]:
import glob, os, shutil, subprocess
DATA = os.path.dirname(glob.glob('/kaggle/input/**/IEMOCAP_features.pkl', recursive=True)[0])
CODE_SRC = os.path.dirname(glob.glob('/kaggle/input/**/extract_wavlm.py', recursive=True)[0])
WAV = os.path.dirname(glob.glob('/kaggle/input/**/Ses01F_impro01_F000.wav', recursive=True)[0])
CODE, OUT = '/kaggle/working/code', '/kaggle/working/wavlm_run'
WAVLM = '/kaggle/working/iemocap_wavlm.pkl'
shutil.copytree(CODE_SRC, CODE, dirs_exist_ok=True)   # /kaggle/input is read-only
print('DATA =', DATA); print('CODE =', CODE)
print('WAV  =', WAV, '|', len(glob.glob(WAV + '/*.wav')), 'wav files (expect 7433)')
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)

### 1. Extract WavLM features (one time, roughly 10-20 min on a T4)
The log shows the layer chosen on the validation split, and an audio-only comparison of IS10 vs WavLM. Note those two probe lines for the report.

In [ ]:
if not os.path.exists(WAVLM):
    proc = subprocess.Popen(['python', 'extract_wavlm.py', '--wav_dir', WAV, '--data_dir', DATA, '--out', WAVLM,
                             '--model', 'microsoft/wavlm-large'], cwd=CODE, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:                 # stream the log into the cell output
        print(line, end='')
    assert proc.wait() == 0, 'extraction failed, see the log above'
print(os.path.getsize(WAVLM) / 1e6, 'MB')

### 2. Sanity run: one WavLM seed

In [ ]:
!cd {CODE} && python train_aux.py --data_dir {DATA} --output_dir {OUT} --tag W_CTRL --seed 1 --audio_feat {WAVLM} | tail -12

### 3. Main sweep: 5 configs x 20 seeds
`CTRL` = original IS10 audio. `W_*` = WavLM audio, without and with the Round 2 losses (λ=1.0, β=0.5, as chosen before).

In [ ]:
!cd {CODE} && python run_sweep.py --data_dir {DATA} --output_dir {OUT} --gpus 0,1 --seeds 1-20 \
    --configs CTRL,W_CTRL,W_VAD,W_SOFT,W_BOTH --audio_feat {WAVLM}

### 4. Summaries
First against the IS10 baseline (does WavLM help?), then against W_CTRL (do the Round 2 losses add anything on top of WavLM?).

In [ ]:
!cd {CODE} && python summarize_aux.py {OUT} CTRL | tee {OUT}/summary_vs_CTRL.txt
!cd {CODE} && python summarize_aux.py {OUT} W_CTRL | tee {OUT}/summary_vs_W_CTRL.txt

### 5. Package results (download from the Output tab)

In [ ]:
!cp {WAVLM} {OUT}/ && cd /kaggle/working && zip -qr wavlm_run.zip wavlm_run && ls -lh wavlm_run.zip